In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import gc

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("C:\\Repos\\Xakaton\\data\\raw")

pd.set_option('display.max_rows', None)      # все строки
pd.set_option('display.max_columns', None)   # все столбцы
pd.set_option('display.width', None)         # не разрывать строки по ширине консоли
pd.set_option('display.max_colwidth', None)  # не обрезать текст внутри ячеек
pd.set_option('display.expand_frame_repr', False) # не переносить широкие таблицы на новые строки

In [2]:
# ============================================================
# 0. Минимальный dfs для восстановлялки
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("C:\\Repos\\Xakaton\\data\\raw")

dfs = {}

dfs["users_courses"] = pd.read_csv(
    DATA_DIR / "users_courses.csv",
    usecols=["user_id", "course_id"]
)

dfs["user_lessons"] = pd.read_csv(
    DATA_DIR / "user_lessons.csv",
    usecols=["user_id", "lesson_id", "group_id", "users_course_id"]
)

dfs["wk_users_courses_actions"] = pd.read_csv(
    DATA_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id"]
)

# Карта колонок для конвертации в целочисленный тип
INT_COLS_MAP = {
    "users_courses": ["user_id", "course_id"],
    "user_lessons": ["user_id", "lesson_id", "group_id", "users_course_id"],
    'wk_users_courses_actions': ['user_id', 'users_course_id']
}

# Конвертация в точности в стиле EDA
for tbl, cols in INT_COLS_MAP.items():
    for col in cols:
        if col not in dfs[tbl].columns:
            continue
        if pd.api.types.is_numeric_dtype(dfs[tbl][col]):
            continue
        dfs[tbl][col] = pd.to_numeric(
            dfs[tbl][col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        ).astype("Int64")

print("Loaded tables:")
for k, v in dfs.items():
    print(k, v.shape)

Loaded tables:
users_courses (290835, 2)
user_lessons (3070664, 4)
wk_users_courses_actions (12909207, 2)


In [3]:
# ============================================================
# 1. Список уникальных user_id и выбор по индексу i
# ============================================================

unique_user_ids = (
    dfs["user_lessons"]["user_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Количество уникальных user_id:", len(unique_user_ids))
display(unique_user_ids.head(20).to_frame(name="user_id"))

i = 0  # меняй вручную
target_user_id = unique_user_ids.iloc[i]

print("i =", i)
print("target_user_id =", target_user_id)

Количество уникальных user_id: 74071


,user_id
0,665743
1,665744
2,665745
3,665768
4,665772
5,665776
6,665778
7,665783
8,665785
9,665786


i = 0
target_user_id = 665743


In [4]:
# ============================================================
# 2. Первые 10 строк user_lessons для выбранного пользователя
# ============================================================

display(
    dfs["user_lessons"]
    .loc[dfs["user_lessons"]["user_id"] == target_user_id]
    .head(10)
)

,user_id,lesson_id,group_id,users_course_id
2176,665743,50004607,50004730,449039


In [5]:
# ============================================================
# 3A. Берем lesson_id и group_id из j-й строки пользователя
# ============================================================

user_lessons_of_target = dfs["user_lessons"].loc[
    dfs["user_lessons"]["user_id"] == target_user_id
].copy()

display(user_lessons_of_target.head(10))

j = 0  # меняй вручную: какую строку пользователя взять
seed_row = user_lessons_of_target.iloc[j]

target_lesson_id = seed_row["lesson_id"]
target_group_id = seed_row["group_id"]

print("target_user_id  =", target_user_id)
print("target_lesson_id =", target_lesson_id)
print("target_group_id  =", target_group_id)

,user_id,lesson_id,group_id,users_course_id
2176,665743,50004607,50004730,449039


target_user_id  = 665743
target_lesson_id = 50004607
target_group_id  = 50004730


In [6]:
# ============================================================
# 3B. Все пользователи с тем же lesson_id и group_id
# ============================================================

same_group_same_lesson = dfs["user_lessons"].loc[
    (dfs["user_lessons"]["lesson_id"] == target_lesson_id) &
    (dfs["user_lessons"]["group_id"] == target_group_id)
].copy()

same_group_user_ids = (
    same_group_same_lesson["user_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Все строки этой группы на этом уроке:")
display(same_group_same_lesson.head(20))

print("Первые 10 user_id из этой группы на этом уроке:")
display(same_group_user_ids.head(10).to_frame(name="user_id"))

Все строки этой группы на этом уроке:


,user_id,lesson_id,group_id,users_course_id
13,665783,50004607,50004730,449109
17,665791,50004607,50004730,449128
18,665793,50004607,50004730,449132
30,665805,50004607,50004730,449161
45,665794,50004607,50004730,449135
49,665827,50004607,50004730,449207
51,666243,50004607,50004730,449904
54,665814,50004607,50004730,449177
56,666235,50004607,50004730,449890
71,665824,50004607,50004730,449202


Первые 10 user_id из этой группы на этом уроке:


,user_id
0,665743
1,665744
2,665776
3,665783
4,665791
5,665793
6,665794
7,665795
8,665796
9,665801


In [7]:
# ============================================================
# 4. Для первых 10 найденных пользователей — их строки из users_courses
# ============================================================

first_10_group_user_ids = same_group_user_ids.head(10).tolist()

users_courses_first_10 = dfs["users_courses"].loc[
    dfs["users_courses"]["user_id"].isin(first_10_group_user_ids)
].copy()

print("Первые 10 user_id группы:")
print(first_10_group_user_ids)

print("Строки users_courses для этих пользователей:")
display(
    users_courses_first_10
    .sort_values(["user_id", "course_id"])
    .head(100)
)

Первые 10 user_id группы:
[665743, 665744, 665776, 665783, 665791, 665793, 665794, 665795, 665796, 665801]
Строки users_courses для этих пользователей:


,user_id,course_id
47783,665743,754
17725,665743,760
58809,665743,50000592
14521,665743,170000688
47760,665744,754
8379,665744,760
83524,665744,931
5928,665744,932
201076,665744,991
134023,665744,1037


In [8]:
# ============================================================
# 4A. Список course_id для каждого из первых 10 пользователей
# ============================================================

courses_per_user_first_10 = (
    users_courses_first_10
    .groupby("user_id")["course_id"]
    .agg(lambda s: sorted(set(s.dropna())))
    .reset_index()
    .rename(columns={"course_id": "course_id_list"})
)

display(courses_per_user_first_10)

,user_id,course_id_list
0,665743,"[754, 760, 50000592, 170000688]"
1,665744,"[754, 760, 931, 932, 991, 1037, 50000592, 170000688]"
2,665776,"[755, 756, 769, 50000592, 170000688]"
3,665783,"[755, 756, 762, 50000592, 170000688]"
4,665791,"[50000592, 170000688]"
5,665793,"[755, 756, 760, 769, 931, 932, 50000592, 170000688]"
6,665794,"[755, 756, 760, 762, 769, 50000592, 170000688]"
7,665795,"[50000592, 170000688]"
8,665796,"[755, 756, 760, 762, 769, 50000592, 170000688]"
9,665801,"[755, 756, 760, 762, 769, 50000592, 170000688]"


In [9]:
# ============================================================
# 5. Пересечение course_id у пользователей найденной группы
# ============================================================

group_users_courses = dfs["users_courses"].loc[
    dfs["users_courses"]["user_id"].isin(same_group_user_ids.tolist())
].copy()

courses_per_user = (
    group_users_courses
    .groupby("user_id")["course_id"]
    .agg(lambda s: set(s.dropna()))
)

if len(courses_per_user) == 0:
    course_intersection = set()
else:
    course_intersection = set.intersection(*courses_per_user.tolist())

course_intersection = sorted(course_intersection)

print("Пересечение course_id у всех пользователей данной группы на данном уроке:")
print(course_intersection)

display(pd.DataFrame({"intersection_course_id": course_intersection}))

Пересечение course_id у всех пользователей данной группы на данном уроке:
[np.int64(50000592)]


,intersection_course_id
0,50000592


In [10]:
# Проверка совпадения множеств user_id в users_courses и user_lessons

uc_user_ids = set(dfs["users_courses"]["user_id"].dropna().unique())
ul_user_ids = set(dfs["user_lessons"]["user_id"].dropna().unique())

only_in_users_courses = sorted(uc_user_ids - ul_user_ids)
only_in_user_lessons = sorted(ul_user_ids - uc_user_ids)

comparison_df = pd.DataFrame([
    {"metric": "users_courses_unique_user_id", "value": len(uc_user_ids)},
    {"metric": "user_lessons_unique_user_id", "value": len(ul_user_ids)},
    {"metric": "intersection_user_id", "value": len(uc_user_ids & ul_user_ids)},
    {"metric": "only_in_users_courses", "value": len(only_in_users_courses)},
    {"metric": "only_in_user_lessons", "value": len(only_in_user_lessons)},
    {"metric": "sets_equal", "value": uc_user_ids == ul_user_ids},
])

display(comparison_df)

print("Первые 20 user_id только в users_courses:")
display(pd.DataFrame({"user_id": only_in_users_courses[:20]}))

print("Первые 20 user_id только в user_lessons:")
display(pd.DataFrame({"user_id": only_in_user_lessons[:20]}))

,metric,value
0,users_courses_unique_user_id,88319
1,user_lessons_unique_user_id,74071
2,intersection_user_id,74039
3,only_in_users_courses,14280
4,only_in_user_lessons,32
5,sets_equal,False


Первые 20 user_id только в users_courses:


,user_id
0,665740
1,665741
2,665742
3,665746
4,665747
5,665748
6,665749
7,665750
8,665751
9,665752


Первые 20 user_id только в user_lessons:


,user_id
0,761581
1,761586
2,761588
3,761599
4,761603
5,761609
6,761614
7,761617
8,761618
9,761652


In [11]:
# Сравнение множеств уникальных users_course_id
# между user_lessons и wk_users_courses_actions

ul_uc_ids = set(dfs["user_lessons"]["users_course_id"].dropna().unique())
wka_uc_ids = set(dfs["wk_users_courses_actions"]["users_course_id"].dropna().unique())

only_in_user_lessons = sorted(ul_uc_ids - wka_uc_ids)
only_in_wk_actions = sorted(wka_uc_ids - ul_uc_ids)

comparison_users_course_id_df = pd.DataFrame([
    {"metric": "user_lessons_unique_users_course_id", "value": len(ul_uc_ids)},
    {"metric": "wk_users_courses_actions_unique_users_course_id", "value": len(wka_uc_ids)},
    {"metric": "intersection_users_course_id", "value": len(ul_uc_ids & wka_uc_ids)},
    {"metric": "only_in_user_lessons", "value": len(only_in_user_lessons)},
    {"metric": "only_in_wk_users_courses_actions", "value": len(only_in_wk_actions)},
    {"metric": "sets_equal", "value": ul_uc_ids == wka_uc_ids},
])

display(comparison_users_course_id_df)

print("Первые 20 users_course_id только в user_lessons:")
display(pd.DataFrame({"users_course_id": only_in_user_lessons[:20]}))

print("Первые 20 users_course_id только в wk_users_courses_actions:")
display(pd.DataFrame({"users_course_id": only_in_wk_actions[:20]}))

,metric,value
0,user_lessons_unique_users_course_id,216482
1,wk_users_courses_actions_unique_users_course_id,216342
2,intersection_users_course_id,216315
3,only_in_user_lessons,167
4,only_in_wk_users_courses_actions,27
5,sets_equal,False


Первые 20 users_course_id только в user_lessons:


,users_course_id
0,451491
1,452078
2,452602
3,454268
4,455420
5,455749
6,456661
7,456668
8,458102
9,459294


Первые 20 users_course_id только в wk_users_courses_actions:


,users_course_id
0,449688
1,451229
2,453344
3,457448
4,459598
5,460297
6,461539
7,462481
8,462906
9,499155
